# Ch.4 — Neural Collaborative Filtering

> **The story.** In **2017**, Xiangnan He and colleagues at the National University of Singapore published "Neural Collaborative Filtering" (_WWW 2017_), arguing that the inner product in matrix factorisation is _too simple_ to capture complex user–item interactions. Their key insight: **replace the dot product with a neural network** that takes user and item embeddings as input and learns an arbitrary interaction function. The architecture — called **NeuMF** — combines two parallel paths: a **Generalised Matrix Factorisation (GMF)** path for linear interactions and a **Multi-Layer Perceptron (MLP)** path for non-linear ones, fused in a final prediction layer. GMF and MLP use _separate_ embedding spaces, letting each path specialise. The paper showed consistent improvements over MF on MovieLens and Pinterest, and launched a wave of deep-learning recommenders at Alibaba, JD.com, and Pinterest. Every "deep collaborative filter" you encounter today traces its lineage to this six-page paper.
>
> **Where you are in the curriculum.** Chapter four of the FlixAI track. Matrix factorisation (Ch.3) achieved **78% HR@10** on MovieLens 100k but is limited to linear interactions ($\hat{r} = \mathbf{u}^\top\mathbf{v}$). Neural CF replaces the dot product with a _learnable_ non-linear function, capturing taste patterns like "loves sci-fi and comedy separately but hates sci-fi comedy hybrids" — interactions the dot product cannot express. This is the **first deep-learning model** in the Recommender Systems track.
>
> **Notation.** $\mathbf{p}_u^G, \mathbf{q}_i^G$ — user/item embeddings in the **GMF** space ($\in \mathbb{R}^d$); $\mathbf{p}_u^M, \mathbf{q}_i^M$ — user/item embeddings in the **MLP** space; $\odot$ — element-wise (Hadamard) product; $\oplus$ — concatenation; $\hat{y}_{ui}$ — predicted interaction probability $\in [0,1]$; $k$ — negative-sampling ratio (default 4).

---

## §0 · The Challenge — Where We Are

> **The mission**: Launch **FlixAI** — >85% HR@10 across 5 constraints: (1) ACCURACY >85%, (2) COLD START, (3) SCALABILITY <200ms, (4) DIVERSITY, (5) EXPLAINABILITY "Because you liked X."

**Progress so far:** Ch.1 → 42%. Ch.2 → 65%. Ch.3 MF → 78%. **Still 7 points short.** MF plateaus at 78% even after increasing factors ($d=8 \to 16 \to 32$, +0.5% each). The architecture itself is the bottleneck: the dot product $\mathbf{u}^\top\mathbf{v}$ is fundamentally linear and cannot encode "likes A and B separately but hates A+B together." Replace it with an MLP that can learn any interaction function.

```mermaid
flowchart LR
 MF["Ch.3: MF\nHR@10 = 78%\nlinear dot product"] --> EMB["Separate GMF\n+ MLP Embeddings"]
 EMB --> GMF["GMF Path\n(linear ⊙)"]
 EMB --> MLP["MLP Path\n(non-linear ReLU)"]
 GMF --> FUSE["Fuse & Predict\nσ(w·GMF ⊕ w·MLP)"]
 MLP --> FUSE
 FUSE --> EVAL["NeuMF\nHR@10 ≈ 82%"]

 style MF fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style EMB fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style FUSE fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style EVAL fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Dataset:** MovieLens 100k | **Task:** Build NeuMF (Neural Matrix Factorization) with PyTorch | **Outcome:** NCF = ~82% HR@10


## §1 · The Core Idea

Matrix factorization predicts ratings with a dot product: if your "sci-fi dimension" is 0.9 and a movie's "sci-fi dimension" is 0.8, you'd rate it highly. But taste is not additive. Consider a user who loves both sci-fi and comedy separately but **hates** sci-fi comedies — the dot product treats dimensions independently and can never capture this cross-dimension interaction.

**Neural CF's answer:** replace the dot product with a neural network. Given user and item as input, an MLP can learn any function of their embeddings — including cross-dimension interactions. The NeuMF architecture uses **two parallel paths** so each can specialise:

- **GMF path** (element-wise product $\odot$): captures linear, dimension-by-dimension alignment — the same signal as MF
- **MLP path** (concatenate → dense layers): captures non-linear cross-dimension interactions

Both paths use _separate_ embedding spaces, letting each specialise. Their outputs are concatenated and passed through a final sigmoid layer.

```mermaid
flowchart LR
    UG["User\nGMF emb p_u^G"] --> GMFOP["⊙ element-wise\nproduct"]
    IG["Item\nGMF emb q_i^G"] --> GMFOP
    UM["User\nMLP emb p_u^M"] --> CONCAT["⊕ concatenate\n→ MLP layers\n(ReLU)"]
    IM["Item\nMLP emb q_i^M"] --> CONCAT
    GMFOP --> FUSE["⊕ fuse → σ(w·x)\nfinal prediction"]
    CONCAT --> FUSE
    FUSE --> OUT["ŷ_ui ∈ [0,1]\ninteraction score"]

    style UG fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style IG fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style UM fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style IM fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style GMFOP fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style CONCAT fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style FUSE fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style OUT fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

Trained on **implicit feedback** (binary: interacted or not) with **negative sampling** — for each positive, sample $k$ items the user didn't interact with as negatives.

> **Optional depth:** $\hat{y}_{ui} = \sigma\!\left(\mathbf{h}^T\!\left[\mathbf{p}_u^G \odot \mathbf{q}_i^G \;\oplus\; \text{MLP}(\mathbf{p}_u^M \oplus \mathbf{q}_i^M)\right]\right)$ — where $\mathbf{h}$ is the learnable fusion weight vector.


In [ ]:
# TODO: Implement this cell
#  (Imports)
#
# Steps:
# 1. Imports
# 2. Compute `SEED` using `set_theme()`
# 3. Compute `device` using `device()`
#
# Hint:
#    device = torch.device(???)

In [ ]:
# TODO: Implement this cell
#  (Load MovieLens 100k)
#
# Steps:
# 1. Load MovieLens 100k
# 2. Compute `ratings` using `read_csv()`
# 3. Compute `n_users`
# 4. Aggregate data into `ratings_sorted` -- use `copy()`
# 5. Aggregate data into `user_rated` -- use `rated()`
# 6. Process data
#
# Hint:
#    ratings = pd.read_csv(???)
#    ratings_sorted = ratings.sort_values(???)
#    test = ratings_sorted.groupby(???)
#    train = ratings_sorted.drop(???)

In [ ]:
# TODO: Implement this cell
#  (Dataset with Negative Sampling)
#
# Steps:
# 1. Dataset with Negative Sampling
# 2. Define helper function
# 3. Call `randint()` to produce the result
# 4. Compute `train_dataset` using `NCFDataset()`
# 5. Process data
#
# Hint:
#    train_dataset = NCFDataset(n_neg=???)
#    train_loader = DataLoader(batch_size=???, shuffle=???)
#    neg_item = np.random.randint(???)

In [ ]:
# TODO: Implement this cell
#  (NeuMF Model)
#
# Steps:
# 1. NeuMF Model
# 2. Call `append()` to produce the result
# 3. Call `Linear()` to produce the result
# 4. Call `_init_weights()` to produce the result
# 5. Define helper function
# 6. Call `cat()` to produce the result
# 7. Compute `model` using `NeuMF()`
#
# Hint:
#    model = NeuMF(d_gmf=???, d_mlp=???)
#    user_gmf = nn.Embedding(???)
#    item_gmf = nn.Embedding(???)
#    user_mlp = nn.Embedding(???)

In [ ]:
# TODO: Implement this cell
#  (Training Loop)
#
# Steps:
# 1. Training Loop
# 2. Compute `n_epochs`
# 3. Call `train()` to produce the result
# 4. Call `to()` to produce the result
# 5. Call `model()` to produce the result
# 6. Call `zero_grad()` to produce the result
# 7. Call `item()` to produce the result
# 8. Call `append()` to produce the result
# 9. Process data
#
# Hint:
#    optimizer = torch.optim.Adam(???)
#    criterion = nn.BCELoss(???)
#    users = users.long(???)
#    items = items.long(???)

In [ ]:
# TODO: Implement this cell
#  (Training Loss Curve)
#
# Steps:
# 1. Training Loss Curve
#
# Hint:
#    ax = plt.subplots(???)

In [ ]:
def hit_rate_at_k(test_df, top_k_per_user, k=10):
    """
    TODO #7: Implement `hit_rate_at_k()`.

    Steps:
    1. Evaluate: HR@10 and NDCG@10
    2. Compute `top_k_ncf` using `no_grad()`
    3. Call `to()` to produce the result
    4. Process data
    5. Call `argsort()` to produce the result
    6. Define helper function `hit_rate_at_k()`
    7. Define helper function `ndcg_at_k()`
    8. Compute `hr_ncf`
    9. Process data

    Hint:
    rated = user_rated.get(???)
    user_tensor = torch.tensor(???)
    item_tensor = torch.arange(???)
    recs = top_k_per_user.get(???)

    Returns: hits / len(test_df)
    """
    raise NotImplementedError("TODO: implement hit_rate_at_k()")


def ndcg_at_k(test_df, top_k_per_user, k=10):
    """
    TODO #7: Implement `ndcg_at_k()`.

    Steps:
    1. Evaluate: HR@10 and NDCG@10
    2. Compute `top_k_ncf` using `no_grad()`
    3. Call `to()` to produce the result
    4. Process data
    5. Call `argsort()` to produce the result
    6. Define helper function `hit_rate_at_k()`
    7. Define helper function `ndcg_at_k()`
    8. Compute `hr_ncf`
    9. Process data

    Hint:
    rated = user_rated.get(???)
    user_tensor = torch.tensor(???)
    item_tensor = torch.arange(???)
    recs = top_k_per_user.get(???)

    Returns: hits / len(test_df)
    """
    raise NotImplementedError("TODO: implement ndcg_at_k()")

In [ ]:
# TODO: Implement this cell
#  (Compare All Methods So Far)
#
# Steps:
# 1. Compare All Methods So Far
# 2. Plot results -- call `subplots()`
# 3. Plot results -- call `text()`
# 4. Plot results -- call `tight_layout()`
#
# Hint:
#    results = pd.DataFrame(???)
#    ax = plt.subplots(???)
#    bars = ax.bar(???)

### What §1 established — and what it still doesn't solve

[Done] **Non-linear interactions learned.** The MLP path captures cross-dimension taste interactions the dot product cannot encode.

[Done] **HR@10 lifted.** From ~78% (Ch.3 MF) to ~82% — closing within 3 points of the 85% target.

[Done] **Implicit feedback.** Trained on binary interaction signals, not just explicit ratings.

**Still open:**

- **Cold start:** New users have no embedding. Without at least one interaction, NeuMF cannot score anything.
- **Content blindness:** Every movie is an opaque integer ID. A new movie with 3 ratings has a near-random embedding.
- **3 points remaining:** The gap is structural — NCF cannot see item content. Ch.5 closes it.


In [ ]:
# TODO: Implement this cell
#  (Embedding Visualisation)
#
# Steps:
# 1. Embedding Visualisation
# 2. Compute `item_emb` using `cpu()`
# 3. Fit the model -- call `PCA()`
# 4. Compute `movies_df` using `read_csv()`
# 5. Compute `genre_cols` using `idxmax()`
# 6. Plot results -- call `subplots()`
# 7. Plot results -- call `Projection()`
#
# Hint:
#    pca = PCA(n_components=???, random_state=???)
#    item_emb = model.item_gmf.weight.data.cpu(???)
#    emb_2d = pca.fit_transform(???)
#    movies_df = pd.read_csv(???)

## Progress Check

**Checkpoint:** FlixAI — hit@10 advanced from ~78% to ~82%, closing within 3 points of the >85% target in this chapter. Non-linear embeddings capture taste interactions that linear matrix factorization cannot.

| #   | Constraint     | Target                | Ch.4 Status                                 |
| --- | -------------- | --------------------- | ------------------------------------------- |
| 1   | ACCURACY       | >85% HR@10            | ~82% — 3 points remaining                   |
| 2   | COLD START     | New users/items       | [No] Embeddings require interaction history |
| 3   | SCALABILITY    | 1M+ ratings           | GPU helpful for training                    |
| 4   | DIVERSITY      | Not just popular      | Richer embeddings help                      |
| 5   | EXPLAINABILITY | "Because you liked X" | Neural network = black box                  |

**Bottom line**: 82% hit rate — just 3 points from target. Non-linear MLP captures taste interactions linear MF missed. But the model is content-blind — it cannot handle new items or reason about genres.

**Next**: Ch.5 — Hybrid Systems → add content features (genres, demographics) to close the final 3-point gap.


## Exercises

**Exercise 1 — GMF-Only vs MLP-Only**
Train GMF-only and MLP-only models separately. Compare HR@10 against the full NeuMF. Which component contributes more?

**Exercise 2 — Negative Sampling Ratio**
Train NeuMF with k=1, 4, and 10 negatives per positive. How does the ratio affect HR@10 and training time?

**Exercise 3 — Pre-Training**
Pre-train GMF and MLP separately, then initialise NeuMF with their weights and fine-tune. Does pre-training improve HR@10?


In [ ]:
# TODO: Implement this cell
#  (Exercise 1 scaffold — GMF-Only vs MLP-Only)
#
# Steps:
# 1. Set up: Exercise 1 scaffold — GMF-Only vs MLP-Only
# 2. Process data
#
# Hint:
#    gmf = self.user_emb(???)

In [ ]:
# TODO: Implement this cell
#  (Exercise 2 scaffold — Negative Sampling Ratio)
#
# Steps:
# 1. Set up: Exercise 2 scaffold — Negative Sampling Ratio
# 2. Process data
#
# Hint:
#    dataset = NCFDataset(n_neg=???)
#    loader = DataLoader(batch_size=???, shuffle=???)

In [ ]:
# TODO: Implement this cell
#  (Exercise 3 scaffold — Pre-Training)
#
# Steps:
# 1. Set up: Exercise 3 scaffold — Pre-Training
# 2. Process data
#
# Hint:
#    gmf_model = GMFOnly(???)
#    mlp_model = MLPOnly(???)